In [ ]:
!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

#semantic search (and outputs 384-dim vectors)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
import numpy as np

def get_embedding(text: str) -> np.ndarray:
    """
    Real embedding function using SentenceTransformers.
    Converts text into a semantic vector for meaning-based search.
    """
    vec = embedder.encode(text, normalize_embeddings=True)  # normalization helps cosine similarity
    return np.array(vec, dtype=float)

In [ ]:
import json #chatgpt recommended
from pathlib import Path

LIB_PATH = Path("book_library.json")

def save_library(book_list, path=LIB_PATH):
    serializable = []
    for b in book_list:
        b2 = dict(b)
        if "embedding" in b2:
            b2["embedding"] = b2["embedding"].tolist()
        serializable.append(b2)
    path.write_text(json.dumps(serializable, ensure_ascii=False, indent=2), encoding="utf-8")

def load_library(path=LIB_PATH):
    if not path.exists():
        return []
    data = json.loads(path.read_text(encoding="utf-8"))
    for b in data:
        if "embedding" in b and b["embedding"] is not None:
            b["embedding"] = np.array(b["embedding"], dtype=float)
    return data

In [ ]:
#Generate and attach embeddings for each book
def add_embeddings_to_books(book_list):
    for book in book_list:
        text = book["summary"] + " " + " ".join(book["tags"])
        book["embedding"] = get_embedding(text)

In [ ]:
def add_book_interactive(book_list):
    print("\nAdd a new book to your library:")
    title = input("Title: ").strip()
    link = input("Link (URL): ").strip()
    tags_raw = input("Tags (comma-separated): ").strip()
    summary = input("Brief plot / what you remember: ").strip()
    rating_raw = input("Rating (0-5): ").strip()
    date_finished = input("Date finished (YYYY-MM-DD, optional): ").strip()

    tags = [t.strip() for t in tags_raw.split(",") if t.strip()]
    try:
        rating = float(rating_raw)
    except:
        rating = 0.0

    book = {
        "title": title or "Untitled",
        "link": link,
        "tags": tags,
        "summary": summary,
        "rating": rating,
        "date_finished": date_finished,
        "revisit_count": 0,
    }

    text = book["summary"] + " " + " ".join(book["tags"])
    book["embedding"] = get_embedding(text)

    book_list.append(book)
    print(f"Added: {book['title']}")

In [ ]:
#Cosine similarity
def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    """
    Compute cosine similarity between two vectors.
    """
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / (norm_a * norm_b))

In [ ]:
#Search books using a vague query
def search_books(query: str, book_list, top_k: int = 5, return_scores: bool = False):
    """
    Search for books that best match the user's vague query.
    Uses embeddings + cosine similarity + a small revisit priority.
    If return_scores=True, returns (book, similarity, boosted_score).
    """
    query_emb = get_embedding(query)
    scored = []

    for book in book_list:
        sim = cosine_similarity(query_emb, book["embedding"])
        boosted_score = sim + 0.05 * book.get("revisit_count", 0)
        scored.append((boosted_score, sim, book))

    scored.sort(key=lambda x: x[0], reverse=True)
    top = scored[:top_k]

    if return_scores:
        return [(b, sim, boosted) for (boosted, sim, b) in top]
    else:
        return [b for (_, _, b) in top]

In [ ]:
def recommend_books(book_list, top_per_tag=2):
    if not book_list:
        print("\nNo books yet. Add a few first!")
        return

    scored = []
    for b in book_list:
        score = float(b.get("rating", 0.0)) + 0.2 * float(b.get("revisit_count", 0))
        scored.append((score, b))
    scored.sort(key=lambda x: x[0], reverse=True)

    tag_map = {}
    for score, b in scored:
        tags = b.get("tags", []) or ["(no tag)"]
        for t in tags:
            tag_map.setdefault(t, [])
            if all(existing["title"] != b["title"] for existing in tag_map[t]):
                tag_map[t].append(b)

    print("\nHello, what would you like to read today?")
    print("Here are your top picks:\n")

    tag_order = sorted(tag_map.keys(), key=lambda t: len(tag_map[t]), reverse=True)
    for t in tag_order:
        picks = tag_map[t][:top_per_tag]
        if not picks:
            continue
        print(f"[{t}]")
        for b in picks:
            print(f" - {b['title']}  (rating {b.get('rating',0)}, revisits {b.get('revisit_count',0)})")
        print()

In [ ]:
#Mark a book as revisited
def mark_revisit(book):
    """
    Increase the revisit count for a book.
    """
    book["revisit_count"] = book.get("revisit_count", 0) + 1

In [ ]:
def refine_query_with_context(original_query, top_books):
    """
    Use information from top matching books to refine the query
    automatically using AI semantics.
    """
    context_texts = []

    for b in top_books[:3]:
        context_texts.append(b["summary"])
        context_texts.extend(b.get("tags", []))

    enriched_query = original_query + " " + " ".join(context_texts)
    return enriched_query

In [ ]:
def run_app():
    global books
    books = load_library()
    if not books:
        books = []

    missing = any("embedding" not in b or b["embedding"] is None for b in books)
    if missing and books:
        add_embeddings_to_books(books)

    while True:
        print("\n=== AI Book Finder ===")
        print("1) Add a book")
        print("2) Search by vague description")
        print("3) Recommendations")
        print("4) Load library")
        print("5) Quit")

        choice = input("Choose an option: ").strip()

        if choice == "1":
            add_book_interactive(books)
            save_library(books)
            print(f"Autosaved to {LIB_PATH}")

        elif choice == "2":
            if not books:
                print("\nYour library is empty. Add a few books first.")
                continue

            query = input("\nDescribe the book you’re looking for: ").strip()
            if not query:
                print("Empty query. Try again.")
                continue

            max_refine = 2  # how many times we ask for extra details

            for attempt in range(max_refine + 1):
                results_scored = search_books(query, books, top_k=5, return_scores=True)

                # enrich the query using AI context
                if attempt > 0:
                    top_books = [b for (b, _, _) in results_scored]
                    query = refine_query_with_context(query, top_books)

                # Print results
                print(f"\nQuery: {query}\n")
                for idx, (book, sim, boosted) in enumerate(results_scored, start=1):
                    print(f"Result {idx}: {book['title']}")
                    print(f"  Similarity: {sim:.3f} | Boosted: {boosted:.3f}")
                    print(f"  Tags: {', '.join(book['tags'])}")
                    print(f"  Rating: {book['rating']}")
                    print(f"  Revisit count: {book['revisit_count']}")
                    print(f"  Link: {book['link']}")
                    print(f"  Summary: {book['summary']}\n")

                # Decide if results look weak or ambiguous (chatgpt recommended)
                top_sim = results_scored[0][1]
                second_sim = results_scored[1][1] if len(results_scored) > 1 else -1.0
                gap = top_sim - second_sim
                low_confidence = top_sim < 0.35
                ambiguous = gap < 0.03
                pick = input("Which result did you open? (number / Enter if none): ").strip()

                # If user picks a result, we record revisit and finish
                if pick.isdigit():
                    idx = int(pick)
                    if 1 <= idx <= len(results_scored):
                        chosen_book = results_scored[idx - 1][0]
                        mark_revisit(chosen_book)
                        save_library(books)  # autosave revisit
                        print(f"Revisit saved for: {chosen_book['title']}")
                        break

                # If user didn't pick anything, try to refine the query
                if attempt < max_refine and (low_confidence or ambiguous):
                    if low_confidence:
                        print("Hmm, I’m not confident I found it. Add 1–2 more specific details.")
                    else:
                        print("These top results are very similar. Add one detail to distinguish them.")

                    extra = input("Extra detail (character / setting / event / trope): ").strip()
                    if extra:
                        query = query + " " + extra
                        continue

                print("\nI couldn’t identify a clear match from your library.")
                print("Try a different description, or add the book into your library first.")
                break

        elif choice == "3":
            recommend_books(books, top_per_tag=2)

        elif choice == "4":
            books = load_library()
            print(f"\nLoaded {len(books)} books:\n")
            for i, b in enumerate(books, start=1):
                print(f"{i}. {b['title']}  (rating {b.get('rating',0)}, revisits {b.get('revisit_count',0)})")


        elif choice == "5":
            save_library(books)
            print("Saved and exiting. Goodbye!")
            break

        else:
            print("Invalid choice. Try again.")


run_app()


=== AI Book Finder ===
1) Add a book
2) Search by vague description
3) Recommendations
4) Load library
5) Quit
Choose an option: 2

Describe the book you’re looking for: girl like regina george but with class and a doctorate degree

Query: girl like regina george but with class and a doctorate degree

Result 1: Underneath a thousand skies
  Similarity: 0.265 | Boosted: 0.465
  Tags: romance, transmigration, mutliple worlds, BE, cunning female lead, system, historical
  Rating: 5.0
  Revisit count: 4
  Link: https://www.wattpad.com/story/186833280-underneath-a-thousand-skies-%E2%9C%94%EF%B8%8F
  Summary: female lead working as a transmigrator and hop into worlds to make sure the male lead doesnt destory each world but he ends up falling in love with her so when she leaves he destorys the world anyways

Result 2: Reborn as the mayor's girlfriend
  Similarity: 0.393 | Boosted: 0.443
  Tags: romance, transmigration, HE, power couple, politics, system
  Rating: 5.0
  Revisit count: 1
  Lin